In [ ]:
import pandas as pd
import os
import json
from pyecharts.charts import Graph
from pyecharts import options as opts
from pyecharts.globals import ThemeType
import re
import difflib
from pyecharts.commons.utils import JsCode
from bs4 import BeautifulSoup as bs
import base64
import numpy as np
import gzip
import io
from PIL import Image
import math

In [ ]:
def convert_png_to_webp(src_directory):
    for root, dirs, files in os.walk(src_directory):
        for file in files:
            if file.endswith('.png'):
                png_file_path = os.path.join(root, file)
                webp_file_path = os.path.splitext(png_file_path)[0] + '.webp'
                if os.path.exists(webp_file_path):
                    continue  # Skip conversion if WEBP file already exists
                # Create the directory for the webp file if it doesn't exist
                os.makedirs(os.path.dirname(webp_file_path), exist_ok=True)

                # Convert PNG to WEBP
                with Image.open(png_file_path) as img:
                    img.save(webp_file_path, 'WEBP', lossless=True)
    print("done")

def circle_points(r, x):
    """
    计算以原点为圆心、半径为 r 的圆上等分点的坐标，起始点正下方，逆时针排列。
    :param r: 圆的半径
    :param x: 等分数
    :return: [(x1, y1), (x2, y2), ...]
    """
    points = []
    for i in range(x):
        angle = 2 * math.pi * i / x - math.pi / 2  # 起始点正下方，逆时针
        px = round(r * math.cos(angle), 2)
        py = round(r * math.sin(angle), 2)
        points.append((-px, -py))
    return points

In [ ]:
GTNH_version = "272"
os.makedirs(f"json/{GTNH_version}/quests_icons/QuestIcon/", exist_ok=True)

convert_png_to_webp(f'quests_icons/{GTNH_version}/')

need_to_replace = ["§%n", "§\n"]

quest_map = {}
quest_line_map = {}
quest_id_set = set()
line_id_set = set()
quest_zh_map = {}
quest_zh_line_map = {}
with open(f"lang/{GTNH_version}/zh_CN.lang", "r", encoding="utf-8") as f:
    for line in f.readlines():
        if "betterquesting.questline" in line and "=" in line:
            one_line = line.split("=")
            quest_line_id = one_line[0].split(".")[-2]
            if "name" in one_line[0]:
                quest_zh_line_map[quest_line_id + "name"] = one_line[1]
            elif "desc" in one_line[0]:
                quest_zh_line_map[quest_line_id + "desc"] = one_line[1]
            line_id_set.add(quest_line_id)
        elif "betterquesting.quest" in line and "=" in line:
            one_line = line.split("=", 1)
            quest_id = one_line[0].split(".")[-2]
            if "name" in one_line[0]:
                quest_zh_map[quest_id + "name"] = one_line[1]
            elif "desc" in one_line[0]:
                quest_zh_map[quest_id + "desc"] = one_line[1]
            quest_id_set.add(quest_id)

quest_name_df = pd.DataFrame(
    columns=["id", "name", "desc", "zh_name", "zh_desc"])
for i in quest_id_set:
    name = quest_map[i + "name"].strip() if (i + "name") in quest_map else ""
    desc = quest_map[i + "desc"].strip() if (i + "desc") in quest_map else ""
    zh_name = quest_zh_map[i + "name"].strip() if (i +
                                                   "name") in quest_zh_map else ""
    zh_desc = quest_zh_map[i + "desc"].strip()if (i +
                                                  "desc") in quest_zh_map else ""

    quest_name_df.loc[len(quest_name_df)] = [i.replace(
        "quest", ""), name, desc, zh_name, zh_desc]

quest_line_name_df = pd.DataFrame(
    columns=["id", "name", "desc", "zh_name", "zh_desc"])
for i in line_id_set:
    name = quest_line_map[i + "name"].strip() if (i +
                                                  "name") in quest_line_map else ""
    desc = quest_line_map[i + "desc"].strip() if (i +
                                                  "desc") in quest_line_map else ""
    zh_name = quest_zh_line_map[i +
                                "name"].strip() if (i + "name") in quest_zh_line_map else ""
    zh_desc = quest_zh_line_map[i +
                                "desc"].strip()if (i + "desc") in quest_zh_line_map else ""
    quest_line_name_df.loc[len(quest_line_name_df)] = [i.replace(
        "line", ""), name, desc, zh_name, zh_desc]

with open(f"DefaultQuests/{GTNH_version}/QuestLinesOrder.txt", "r") as f:
    en_quest_line = f.readlines()

quest_line_json = []
quest_line_icons_dict = {}
for l in en_quest_line:
    quest_line_id = re.sub("[^a-zA-Z0-9_]", "", l.split(":")[0])
    quest_line_name = l.split(":")[1].strip()
    line = quest_line_name_df[quest_line_name_df["id"] == quest_line_id]

    shorter_name = re.sub("[^a-zA-Z0-9]", "", quest_line_name)[:16]
    quest_line_json.append({
        "title": quest_line_name,
        "title_zh": line["zh_name"].values[0],
        "quest": shorter_name,
    })

    with open(f"quests_icons/{GTNH_version}/QuestLineIcon/{shorter_name}.webp", "rb") as img:
        quest_line_icons_dict[shorter_name] = base64.b64encode(
            img.read()).decode()

with open(f"json/{GTNH_version}/quest_line.json", "w", encoding="utf8") as f:
    f.write(str(quest_line_json).replace("'", "\""))

with open(f"json/{GTNH_version}/quests_icons/QuestLineIcon.gtbl", "wb") as f:
    f.write(gzip.compress(json.dumps(
        quest_line_icons_dict).encode("utf-8")))

all_icon_map = {"QuestLineIcon": [i["quest"] for i in quest_line_json]}

allInOneLinks = []
allInOneQuestLineMap = {}
for ver in ["zh", "en"]:
    df = pd.DataFrame(columns=["quest_file_name", "quest_id",
                               "quest_name", "quest_desc", "quest_icon", "pre_quests", "quest_name_with_format", "is_main"])
    for quest_line in os.listdir(f"DefaultQuests/{GTNH_version}/Quests/"):
        for quest in os.listdir(f"DefaultQuests/{GTNH_version}/Quests/{quest_line}/"):
            que = json.load(
                open(f"DefaultQuests/{GTNH_version}/Quests/{quest_line}/{quest}", "r", encoding="utf-8"))

            quest_high_id = que["questIDHigh:4"] if "questIDHigh:4" in que else 0
            quest_low_id = que["questIDLow:4"]
            quest_name = que["properties:10"]["betterquesting:10"]["name:8"]
            quest_desc = que["properties:10"]["betterquesting:10"]["desc:8"]
            icon_name = str(quest_low_id)
            is_main = que["properties:10"]["betterquesting:10"]["isMain:1"]

            pre_quests = []
            for i in que["preRequisites:9"]:
                pre_quests.append(que["preRequisites:9"][i]["questIDLow:4"])

            str_id = base64.urlsafe_b64encode(bytes.fromhex((quest_high_id.to_bytes(length=8, signed=True, byteorder="big") +
                                                             quest_low_id.to_bytes(length=8, signed=True, byteorder="big")).hex())).decode("utf-8").replace("==", "")
            if ver == "zh":
                line = quest_name_df[quest_name_df["id"] == str_id]
                if len(line) != 0:
                    quest_name = line["zh_name"].values[0]
                    quest_desc = line["zh_desc"].values[0].replace("%%", "%")

            for s in need_to_replace:
                quest_name = quest_name.replace(s, "")
                quest_desc = quest_desc.replace(s, "")

            quest_desc = quest_desc.replace("%n", "<br/>")
            quest_desc = quest_desc.replace("\n", "<br/>")

            df.loc[len(df)] = [quest, quest_low_id, re.sub("(§.)", "", quest_name.strip()),
                               quest_desc, icon_name, pre_quests, quest_name, is_main]

    repeats = list(set(df[df["quest_name"].duplicated()]["quest_name"]))
    for word in repeats:
        counter = 0
        for i in df[df["quest_name"] == word].sort_values("quest_id").index:
            df.loc[i, "quest_name"] = df.loc[i, "quest_name"] + ("\u200B" * counter)
            counter += 1

    # ========================================== #

    whole_json = {}

    for questline in os.listdir(f"DefaultQuests/{GTNH_version}/QuestLines"):
        max_x = -99999
        min_x = 999999
        max_y = -99999
        min_y = 999999
        questLineShorterName = questline.split("-")[0].strip()

        nodes = []
        links = []
        max_size = 0
        node_size = []
        quests_icons_map = {}
        icon_list = []
        for quest in os.listdir(f"DefaultQuests/{GTNH_version}/QuestLines/{questline}"):
            if quest == "QuestLine.json":
                continue

            que = json.load(
                open(f"DefaultQuests/{GTNH_version}/QuestLines/{questline}/{quest}", "r", encoding="utf-8"))
            line = df[df["quest_id"] == que["questIDLow:4"]]
            desc = line["quest_desc"].values[0]

            if ver == "zh":
                with open(f"quests_icons/{GTNH_version}/QuestIcon/{line['quest_icon'].values[0]}.webp", "rb") as img:
                    quests_icons_map[line['quest_icon'].values[0]
                                     ] = base64.b64encode(img.read()).decode()
                icon_list.append(line['quest_icon'].values[0])

            name = line["quest_name"].values[0]

            if "questIDHigh:4" in que:
                high_id = que["questIDHigh:4"]
            else:
                high_id = 0
            low_id = que["questIDLow:4"]

            quest_id_str = base64.urlsafe_b64encode(bytes.fromhex((high_id.to_bytes(
                length=8, signed=True, byteorder="big") + low_id.to_bytes(length=8, signed=True, byteorder="big")).hex())).decode("utf-8")

            title = line["quest_name_with_format"].values[0]

            is_main = line["is_main"].values[0]

            new_node = {"name": name,
                        "symbolSize": int(que["sizeX:3"]),
                        "symbol": "",
                        "x": (que["x:3"] + int(que["sizeX:3"]) / 2),
                        "y": (que["y:3"] + int(que["sizeX:3"]) / 2),
                        "data": desc,
                        "quest_id": quest_id_str,
                        "title": title,
                        "is_main": 1 if is_main else 0,
                        "tooltip": {"show": True,}
                        }

            max_x = max(max_x, new_node["x"])
            min_x = min(min_x, new_node["x"])
            max_y = max(max_y, new_node["y"])
            min_y = min(min_y, new_node["y"])


            pre_quest_names = []
            pre_quests = line["pre_quests"].values[0]
            for quest_id in pre_quests:
                pre_que = df[df["quest_id"] == quest_id]
                pre_name = pre_que["quest_name"].values[0]
                pre_quest_names.append(pre_name)

                temp_link = {"source": pre_name,
                             "target": name,
                             "symbol": ["none", "arrow"], "lineStyle": {"width": 2}, "symbolSize": [0, 10]}
                allInOneLink = {"source": pre_name,
                                "target": name,
                                "symbol": ["none", "arrow"], "lineStyle": {"width": 1}, "symbolSize": [0, 5]}
                if int(pre_que["is_main"].iloc[0]) == int(is_main) == 1:
                    temp_link["lineStyle"] = {"color": "#00c800", "width": 4}
                    allInOneLink["lineStyle"] = {"color": "#00c800", "width": 2}
                links.append(temp_link)
                allInOneLinks.append(allInOneLink)
            if ver == "zh":
                new_node["tooltip"] = f"<strong>{name}</strong>"
                if len(pre_quest_names) > 0:
                    new_node["tooltip"] += "<br/>前置任务需求:<br/>" + \
                        "<br/>".join(pre_quest_names)
            elif ver == "en":
                new_node["tooltip"] = f"<strong>{name}</strong>"
                if len(pre_quest_names) > 0:
                    new_node["tooltip"] += "<br/>Prequest requests:<br/>" + \
                        "<br/>".join(pre_quest_names)

            node_size.append(new_node["symbolSize"] * 1.3)
            nodes.append(new_node)

        
        if max_x - min_x >= 700:
            x_multi = 700 / (max_x - min_x)
        else:
            x_multi = 1
        
        if max_y - min_y >= 470:
            y_multi = 470 / (max_y - min_y)
        else:
            y_multi = 1

        mid_x = (max_x + min_x) / 2
        mid_y = (max_y + min_y) / 2
        newNodeList = []
        for n in nodes:
            t = n.copy()
            t["x"] = round((t["x"] - mid_x) * x_multi, 2)
            t["y"] = round((t["y"] - mid_y) * y_multi, 2)
            t["symbolSize"] = round(t["symbolSize"] / 10, 2)
            newNodeList.append(t)

        allInOneQuestLineMap[questLineShorterName] = newNodeList

        # 调整所有任务图标大小
        avg = sum(node_size) / len(node_size)
        if sum(node_size) <= 1000:
            multi = 1.8
        else:
            multi = 37 / avg
        for node in nodes:
            node["symbolSize"] = round(node["symbolSize"] * multi, 2)

        print(
            f"questline {questline.split("-", 1)[0]},\n there has {len(node_size)} nodes, node size sum {"%.2f" % sum(node_size)}, avg size is {"%.2f" % avg}, multi is{"%.2f" % multi}")

        if ver == "zh":
            with open(f"json/{GTNH_version}/quests_icons/QuestIcon/{questline.split("-", 1)[0]}.gtbl", "wb") as f:
                f.write(gzip.compress(json.dumps(
                    quests_icons_map).encode("utf-8")))
            all_icon_map[f"QuestIcon/{questline.split("-", 1)[0]}"] = icon_list

        g = (
            Graph()
            .add("", nodes, links)
        )
        series = g.options["series"][0]
        whole_json[questline.split(
            '-')[0]] = {"data": series["data"], "links": series["links"]}

    with open(f"json/{GTNH_version}/quest_json{"" if ver == "zh" else "_en"}.json", "w", encoding="utf-8") as quest_json:
        json.dump(whole_json, quest_json,
                  separators=(',', ':'), ensure_ascii=False)
    if ver == "zh":
        with open(f"json/{GTNH_version}/quests_icons.json", "w", encoding="utf8") as f:
            json.dump(all_icon_map, f)
    

    # for all in one
    # ====================

    multipleQuestLinesQuests = []
    for i in os.listdir(f"DefaultQuests/{GTNH_version}/Quests/MultipleQuestLine"):
        multipleQuestLinesQuests.append(i.split("-")[1].replace(".json", ""))

    mainQuestLineNames = ["", 'AndSoItBegins', 'Tier0StoneAge', 'Tier05Steam', 'Tier1LV', 'Tier2MV', 'Tier3HV', 'Tier4EV', 'Tier5IV', 'Tier6LuV',
        'Tier7ZPM', 'Tier8UV', 'Tier9UHV', 'Tier10UEV', 'Tier11UIV', 'Tier12UMV', 'EndgameGoals']
    base_position = circle_points(2500, len(mainQuestLineNames))
    nodeNames = set()
    allNodesLists = []
    for i in range(len(mainQuestLineNames)):
        if mainQuestLineNames[i] == '':
            continue
        nodes = allInOneQuestLineMap[mainQuestLineNames[i]]
        questLineNodes = []
        for n in nodes:
            if n["name"] not in nodeNames:
                nodeNames.add(n["name"])
            else:
                print(f"Duplicate node name found: {n['name']}")
                continue
            n["x"] = round(n["x"] + base_position[i][0], 2)
            n["y"] = round(n["y"] + base_position[i][1], 2)
            questLineNodes.append(n)
        allNodesLists.append(questLineNodes)

    # allNodesLists.append(
    #{"name": "DM is GOD",
    #                 "symbolSize": 20,
    #                 "x": 0,
    #                 "y": 0,
    #                 "symbol": "image://dm.png"
    #                 })
    with open(f"json/{GTNH_version}/allInOne{"" if ver == "zh" else "_en"}.json", "w", encoding="utf-8") as allInOneJson:
        json.dump({"links": allInOneLinks,
                   "datas": {"questNames": mainQuestLineNames[1:], "allNodesLists": allNodesLists}}, allInOneJson,
                  separators=(',', ':'), ensure_ascii=False, indent=4)

In [ ]:
allInOneLinks = []
allInOneQuestLineMap = {}
quests_icons_map = {}
for l in en_quest_line:
    nodes = []
    max_x = -99999
    min_x = 999999
    max_y = -99999
    min_y = 999999
    quest_line_id = re.sub("[^a-zA-Z0-9_]", "", l.split(":")[0])
    quest_line_name = l.split(":")[1].strip()
    line = quest_line_name_df[quest_line_name_df["id"] == quest_line_id]

    shorter_name = re.sub("[^a-zA-Z0-9]", "", quest_line_name)[:16]
    # print(f"{shorter_name}-{l.split(":")[0]}")
    for quest in os.listdir(f"DefaultQuests/{GTNH_version}/QuestLines/{shorter_name}-{l.split(":")[0]}"):
        if quest == "QuestLine.json":
            continue

        que = json.load(
            open(f"DefaultQuests/{GTNH_version}/QuestLines/{shorter_name}-{l.split(':')[0]}/{quest}", "r", encoding="utf-8"))
        line = df[df["quest_id"] == que["questIDLow:4"]]
        desc = line["quest_desc"].values[0]

        name = line["quest_name"].values[0]

        if "questIDHigh:4" in que:
            high_id = que["questIDHigh:4"]
        else:
            high_id = 0
        low_id = que["questIDLow:4"]

        quest_id_str = base64.urlsafe_b64encode(bytes.fromhex((high_id.to_bytes(
            length=8, signed=True, byteorder="big") + low_id.to_bytes(length=8, signed=True, byteorder="big")).hex())).decode("utf-8")

        title = line["quest_name_with_format"].values[0]

        is_main = line["is_main"].values[0]

        new_node = {"name": name,
                    "symbolSize": int(que["sizeX:3"]) / 10,
                    # "symbol": "",
                    "x": (que["x:3"] + int(que["sizeX:3"]) / 2),
                    "y": (que["y:3"] + int(que["sizeX:3"]) / 2),
                    "data": desc,
                    "quest_id": quest_id_str,
                    "title": title,
                    "is_main": 1 if is_main else 0,
                    "tooltip": {
                        "show": True,
                    },

                    }

        max_x = max(max_x, new_node["x"])
        min_x = min(min_x, new_node["x"])
        max_y = max(max_y, new_node["y"])
        min_y = min(min_y, new_node["y"])

        pre_quest_names = []
        pre_quests = line["pre_quests"].values[0]
        for quest_id in pre_quests:
            pre_que = df[df["quest_id"] == quest_id]
            pre_name = pre_que["quest_name"].values[0]
            pre_quest_names.append(pre_name)

            temp_link = {"source": pre_name,
                            "target": name,
                            "symbol": ["none", "arrow"], "lineStyle": {"width": 1}, "symbolSize": [0, 5]}
            if int(pre_que["is_main"].iloc[0]) == int(is_main) == 1:
                temp_link["lineStyle"] = {"color": "#00c800", "width": 2}
            allInOneLinks.append(temp_link)

        if ver == "zh":
            new_node["tooltip"] = f"<strong>{name}</strong>"
            if len(pre_quest_names) > 0:
                new_node["tooltip"] += "<br/>前置任务需求:<br/>" + \
                    "<br/>".join(pre_quest_names)
        elif ver == "en":
            new_node["tooltip"] = f"<strong>{name}</strong>"
            if len(pre_quest_names) > 0:
                new_node["tooltip"] += "<br/>Prequest requests:<br/>" + \
                    "<br/>".join(pre_quest_names)
        nodes.append(new_node)

    if max_x - min_x >= 700:
        x_multi = 700 / (max_x - min_x)
    else:
        x_multi = 1
    
    if max_y - min_y >= 470:
        y_multi = 470 / (max_y - min_y)
    else:
        y_multi = 1

    mid_x = (max_x + min_x) / 2
    mid_y = (max_y + min_y) / 2
    for n in nodes:
        n["x"] = round((n["x"] - mid_x) * x_multi, 2)
        n["y"] = round((n["y"] - mid_y) * y_multi, 2)

    allInOneQuestLineMap[shorter_name] = nodes

multipleQuestLinesQuests = []
for i in os.listdir(f"DefaultQuests/{GTNH_version}/Quests/MultipleQuestLine"):
    multipleQuestLinesQuests.append(i.split("-")[1].replace(".json", ""))

mainQuestLineNames = ["", 'AndSoItBegins', 'Tier0StoneAge', 'Tier05Steam', 'Tier1LV', 'Tier2MV', 'Tier3HV', 'Tier4EV', 'Tier5IV', 'Tier6LuV',
     'Tier7ZPM', 'Tier8UV', 'Tier9UHV', 'Tier10UEV', 'Tier11UIV', 'Tier12UMV', 'EndgameGoals']
base_position = circle_points(2500, len(mainQuestLineNames))
nodeNames = set()
allNodesLists = []
tempList = []
for i in range(len(mainQuestLineNames)):
    if mainQuestLineNames[i] == '':
        continue
    nodes = allInOneQuestLineMap[mainQuestLineNames[i]]
    z = []
    for n in nodes:
        if n["name"] not in nodeNames:
            nodeNames.add(n["name"])
        else:
            print(f"Duplicate node name found: {n['name']}")
            continue

        n["x"] = round(n["x"] + base_position[i][0], 2)
        n["y"] = round(n["y"] + base_position[i][1], 2)
        allNodesLists.append(n)
        z.append(n)
    tempList.append(z)

allNodesLists.append({"name": "DM is GOD",
                 "symbolSize": 20,
                 "x": 0,
                 "y": 0,
                 "symbol": "image://dm.png"
                 })

g = (
    Graph(init_opts=opts.InitOpts(width="100%",
                                  height="980%", bg_color="#f5f0d3", chart_id="this_chart", ))
    .add("t", [], allInOneLinks, is_draggable=False, edge_symbol=['circle', 'arrow'], repulsion=0, label_opts=opts.LabelOpts(position="up", is_show=False), gravity=0, layout="none", edge_symbol_size=10)
).render()

In [ ]:
with open("test.json", "w", encoding="utf-8") as f:
    f.write(json.dumps({"data": tempList}, ensure_ascii=False))